# Module 02 — Polars

**Formation Big Data — ANSD / Data Innovation Lab**

Premier des trois outils du bloc. Polars est écrit en Rust, utilise le format
colonnaire Arrow, exploite **tous les cœurs** de la machine et sait décrire un
calcul avant de l'exécuter, pour l'optimiser.

Au programme :

1. les **expressions** — la façon dont Polars décrit un calcul ;
2. les opérations courantes, en miroir de pandas ;
3. le **mode paresseux** et la lecture du plan optimisé — le moment fort ;
4. le **moteur de flux**, pour dépasser la mémoire disponible ;
5. la campagne de mesures, à reporter dans le comparatif.

## 1. Protocole et contexte

In [ ]:
%load_ext autoreload 
%autoreload 2
import sys
from pathlib import Path
try:
    sys.path.append(str(Path(__file__).parent.parent.resolve()))
except NameError:
    sys.path.append(str(Path.cwd().parent.resolve()))
from tools.outils_mesure import (FICHIER, FICHIER_REGIONS, VOLUME_COMPARAISON,
                           afficher_protocole, contexte_machine, enregistrer,
                           memoire_mo, mesurer)

import polars as pl

print("Polars", pl.__version__)
print()
contexte_machine()

Un cœur physique est une véritable unité de calcul présente dans le processeur, tandis qu’un cœur logique est un processeur virtuel créé par une technologie comme l’Hyper-Threading pour permettre à un cœur physique de gérer plusieurs tâches simultanément.

In [ ]:
afficher_protocole()

> ⚠️ **Le nombre de cœurs conditionne vos résultats.** Sur une machine à un
> ou deux cœurs, Polars ne pourra pas montrer son avantage principal. Notez
> votre configuration : elle servira à interpréter le comparatif final.

## 2. Premier contact

Polars lit un CSV comme pandas, avec une différence visible immédiatement :
le type de chaque colonne est affiché sous son nom.

In [ ]:
df = pl.read_csv(FICHIER, n_rows=VOLUME_COMPARAISON)
df.head()

In [ ]:
print(df.shape)
print()
print(df.schema)

In [ ]:
# Statistiques descriptives
df.describe()

## 3. Les expressions : la grande différence avec pandas

En pandas, on manipule directement des objets : `df["age"].mean()` calcule
tout de suite. En Polars, `pl.col("age").mean()` ne calcule rien — c'est une
**description de calcul**, que l'on confie ensuite au moteur.

C'est ce décalage qui déroute pendant cinq minutes, puis qui explique tout :
parce que Polars reçoit une description et non une suite d'ordres, il peut la
réorganiser avant de l'exécuter.

In [ ]:
# Une expression, isolée : elle ne calcule rien
expression = pl.col("age").mean()
expression

In [ ]:
# La même expression, confiée au moteur
df.select(pl.col("age").mean())

In [ ]:
# Plusieurs expressions d'un coup, exécutées en parallèle
df.select(
    pl.len().alias("effectif"),
    pl.col("age").mean().alias("age_moyen"),
    pl.col("age").max().alias("age_max"),
    pl.col("region").n_unique().alias("nb_libelles_region"),
)

### Table de correspondance pandas → Polars

Gardez cette table : c'est votre antisèche de traduction.

| Opération | pandas | Polars |
|---|---|---|
| Sélectionner des colonnes | `df[["a", "b"]]` | `df.select("a", "b")` |
| Filtrer | `df[df.age >= 15]` | `df.filter(pl.col("age") >= 15)` |
| Créer une colonne | `df["x"] = df.a * 2` | `df.with_columns((pl.col("a") * 2).alias("x"))` |
| Agréger | `df.groupby("r")["a"].mean()` | `df.group_by("r").agg(pl.col("a").mean())` |
| Trier | `df.sort_values("a")` | `df.sort("a")` |
| Joindre | `df.merge(autre, on="k")` | `df.join(autre, on="k")` |
| Compter | `len(df)` | `df.height` ou `pl.len()` |
| Valeurs manquantes | `df.isna().sum()` | `df.null_count()` |

Les utilisateurs de R reconnaîtront la logique de `dplyr` : un verbe par
opération, des expressions composables.

In [ ]:
# Nettoyage des libellés de région, en Polars
print("Avant :", df["region"].n_unique(), "libellés")

df_propre = df.with_columns(
    pl.col("region").str.strip_chars().str.to_titlecase()
)

print("Après :", df_propre["region"].n_unique(), "libellés")

## 4. Les trois calculs métier

Les mêmes qu'au notebook 01, pour comparer l'expressivité.

In [ ]:
# Effectifs par région, du plus peuplé au moins peuplé
effectifs = (df_propre
             .group_by("region")
             .agg(pl.len().alias("effectif"))
             .sort("effectif", descending=True))
effectifs.head()

In [ ]:
# Âge moyen par sexe, en écartant les âges impossibles
(df_propre
 .filter(pl.col("age").is_between(0, 110))
 .group_by("sexe")
 .agg(pl.col("age").mean().round(1).alias("age_moyen"))
 .sort("sexe"))

In [ ]:
# Taux d'activité par milieu de résidence, chez les 15 ans et plus.
# `is_in(...).mean()` renvoie directement une proportion : on la met en %.
(df_propre
 .filter(pl.col("age").is_between(15, 110))
 .group_by("milieu_residence")
 .agg(
     (100 * pl.col("situation_activite")
      .is_in(["Occupé", "Chômeur"]).mean()).round(1).alias("taux_activite"),
     pl.len().alias("effectif"),
 )
 .sort("milieu_residence"))

## 5. Le mode paresseux — le cœur du sujet

Jusqu'ici, chaque instruction s'est exécutée immédiatement. Passons en mode
**paresseux** : on décrit toute la chaîne de traitement, et rien ne s'exécute
avant qu'on ne réclame le résultat.

- `pl.scan_csv(...)` au lieu de `pl.read_csv(...)` : **aucune lecture**
- on enchaîne les opérations : toujours aucun calcul
- `.collect()` : le moteur optimise le plan, puis l'exécute

In [ ]:
plan = (
    pl.scan_csv(FICHIER)
      .with_columns(pl.col("region").str.strip_chars().str.to_titlecase())
      .filter(pl.col("age").is_between(0, 110))
      .filter(pl.col("age") >= 15)
      .group_by("region")
      .agg(
          pl.len().alias("effectif"),
          pl.col("age").mean().alias("age_moyen"),
      )
      .sort("effectif", descending=True)
)

# Rien n'a encore été lu ni calculé : `plan` n'est qu'une description.
type(plan)

### Lecture du plan optimisé

C'est la cellule la plus instructive du notebook. Comparez ce que vous avez
**écrit** et ce que le moteur a **décidé de faire**.

In [ ]:
print(plan.explain())

**Question 1.** Dans le plan ci-dessus, repérez la ligne qui commence par
`PROJECT`. Combien de colonnes le moteur va-t-il réellement lire, sur les 21 du
fichier ? Pourquoi ?

*Votre réponse :* …

**Question 2.** Repérez la ligne `SELECTION`. À quel moment le filtre sur l'âge
est-il appliqué : après la lecture, comme vous l'avez écrit, ou pendant ?
Quelle économie cela représente-t-il ?

*Votre réponse :* …

**Question 3.** Vous avez écrit deux `.filter()` successifs. Combien en
reste-t-il dans le plan ?

*Votre réponse :* …

In [ ]:
# Exécution effective du plan
resultat = plan.collect()
resultat

### Ce que cela change, chiffré

Comparons deux façons d'obtenir **le même résultat** : en chargeant tout puis
en calculant (le réflexe pandas), ou en laissant le moteur optimiser.

In [ ]:
# Lecture à blanc, pour que le cache du système soit dans le même état
# pour les deux mesures qui suivent.
_ = pl.scan_csv(FICHIER).select(pl.len()).collect()

print("Approche « je charge tout, puis je calcule » :")
mesure_immediate = mesurer("immédiat", lambda: (
    pl.read_csv(FICHIER)
      .with_columns(pl.col("region").str.strip_chars().str.to_titlecase())
      .filter(pl.col("age").is_between(0, 110))
      .filter(pl.col("age") >= 15)
      .group_by("region")
      .agg(pl.len().alias("effectif"), pl.col("age").mean().alias("age_moyen"))
))

print("\nApproche « je décris, le moteur optimise » :")
mesure_paresseuse = mesurer("paresseux", lambda: plan.collect())

facteur = mesure_immediate["secondes"] / mesure_paresseuse["secondes"]
print(f"\nFacteur d'accélération : ×{facteur:.1f}")

**Question 4.** D'où vient l'essentiel de ce gain : de la rapidité du
moteur, ou de la réorganisation du plan ? Quelle conséquence pour la façon
d'écrire vos traitements ?

*Votre réponse :* …

C'est la distinction du protocole : à **travail identique**, les moteurs se
tiennent ; à **objectif identique**, celui qui optimise le plan prend une
avance considérable.

## 6. Le moteur de flux

Le mode paresseux permet aussi de traiter des données **plus volumineuses que
la mémoire disponible** : le moteur lit par morceaux, agrège au fur et à
mesure, et n'a jamais besoin de tout garder.

In [ ]:
# Le même plan, exécuté en flux
resultat_flux = plan.collect(engine="streaming")
resultat_flux.head(3)

In [ ]:
# Mesurez le pic de mémoire des deux moteurs sur le fichier
# COMPLET (sans limite de lignes), et comparez.
#

m_memoire = mesurer("moteur en mémoire", lambda: plan.collect(engine="in-memory"))
m_flux    = mesurer("moteur en flux",    lambda: plan.collect(engine="streaming"))

print(f"\nMémoire supplémentaire — en mémoire : {m_memoire['surcout_memoire_mo']} Mo"
      f"  |  en flux : {m_flux['surcout_memoire_mo']} Mo")
print(f"Temps — en mémoire : {m_memoire['secondes']} s"
      f"  |  en flux : {m_flux['secondes']} s")

**Question 5.** Le moteur de flux est-il plus rapide ? Consomme-t-il moins
de mémoire ? Dans quelle situation le choisiriez-vous ?

*Votre réponse :* …

> Nuance importante : le traitement en flux fonctionne bien pour les filtres et
> les agrégations. Un **tri global** ou une **jointure volumineuse** obligent le
> moteur à voir toutes les données, et donc à déborder sur le disque. Le flux ne
> supprime pas le problème, il le déplace.

## 7. Campagne de mesures

Nous mesurons maintenant les cinq opérations de référence, dans l'ordre fixé
par le protocole, sur le volume de comparaison. Ces mesures alimenteront le
comparatif final.

In [ ]:
# Fourni : préparation des données de la campagne
df_mesure = pl.read_csv(FICHIER, n_rows=VOLUME_COMPARAISON)
reference = df_mesure.select("id_individu", "nom").sample(fraction=0.5, seed=1)
print(f"{df_mesure.height:,} lignes chargées".replace(",", " "))

In [ ]:
# Campagne de mesures : cinq opérations, dans l'ordre du protocole
mesures = []
mesures.append(mesurer("lecture",
                       lambda: pl.read_csv(FICHIER, n_rows=VOLUME_COMPARAISON)))
mesures.append(mesurer("filtre",
                       lambda: df_mesure.filter(pl.col("age") >= 15)))
mesures.append(mesurer("agregation",
                       lambda: df_mesure.group_by("region")
                                        .agg(pl.col("age").mean())))
mesures.append(mesurer("tri",
                       lambda: df_mesure.sort(["region", "age"])))
mesures.append(mesurer("jointure",
                       lambda: df_mesure.join(reference, on="id_individu",
                                              how="left")))

enregistrer(mesures, outil="polars", volume="2M", lignes=VOLUME_COMPARAISON)

### Passe sur le fichier complet

Cette fois sans limite de lignes. Certaines opérations peuvent échouer : c'est
une mesure comme une autre, elle sera enregistrée telle quelle.

In [ ]:
# Les mêmes cinq mesures, sur le fichier complet.
# Un échec éventuel est capturé par `mesurer` et enregistré : c'est un résultat.

del df_mesure, reference
import gc; gc.collect()

df_complet = pl.read_csv(FICHIER)
reference_complete = df_complet.select("id_individu", "nom").sample(fraction=0.5, seed=1)
print(f"{df_complet.height:,} lignes".replace(",", " "))

mesures_completes = []
mesures_completes.append(mesurer("lecture",
                                 lambda: pl.read_csv(FICHIER)))
mesures_completes.append(mesurer("filtre",
                                 lambda: df_complet.filter(pl.col("age") >= 15)))
mesures_completes.append(mesurer("agregation",
                                 lambda: df_complet.group_by("region")
                                                   .agg(pl.col("age").mean())))
mesures_completes.append(mesurer("tri",
                                 lambda: df_complet.sort(["region", "age"])))
mesures_completes.append(mesurer("jointure",
                                 lambda: df_complet.join(reference_complete,
                                                         on="id_individu",
                                                         how="left")))

enregistrer(mesures_completes, outil="polars", volume="complet",
            lignes=df_complet.height)

## 8. Ce qu'il faut retenir

- Polars décrit les calculs sous forme d'**expressions**, ce qui lui permet de
  les réorganiser avant de les exécuter.
- Le **mode paresseux** (`scan_csv` … `collect`) est le mode à privilégier :
  c'est lui qui apporte l'essentiel du gain, en ne lisant que les colonnes
  utiles et en filtrant pendant la lecture.
- `.explain()` affiche le plan retenu : c'est l'outil de diagnostic à connaître.
- Le **moteur de flux** permet de dépasser la mémoire disponible, avec des
  limites sur les tris et les jointures.
- La syntaxe est proche de `dplyr` : un atout pour une équipe formée sous R.

**À compléter :**

- Cœurs de ma machine : …
- Facteur immédiat → paresseux : ×…
- Opération la plus rapide : … · la plus lente : …
- Échecs sur le fichier complet : …
